# Fine-tune Qwen2.5-3B on Singapore Criminal Law Q&A

**SUTD MLOps Group 7** — Training notebook for the Singapore Criminal Law Advisory System.

Uses QLoRA (4-bit quantization + LoRA adapters) via [Unsloth](https://github.com/unslothai/unsloth) for fast, memory-efficient fine-tuning.

Logs training metrics to Weights & Biases.

**Runtime:** ~1.5–2 hours on Colab T4  
**GPU required:** T4 (16 GB VRAM) — set via Runtime → Change runtime type → T4 GPU  
> 7B cannot train on T4 even with QLoRA (model alone uses 14+ GB). 3B is the practical maximum for a free T4.

## 0. Install dependencies

In [ ]:
%%capture
# Unsloth — fast QLoRA fine-tuning (must be installed before transformers)
!pip install unsloth
!pip install wandb datasets

## 1. Configuration

In [ ]:
# ── Training config ────────────────────────────────────────────────────────
BASE_MODEL     = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # 3B max practical size for T4 16 GB
OUTPUT_DIR     = "sg-law-qwen2.5-3b-lora"
MAX_SEQ_LEN    = 1024

# LoRA hyperparameters
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05

# Training hyperparameters — tuned for T4 16 GB free tier (2-hr limit)
BATCH_SIZE     = 2
GRAD_ACCUM     = 8    # effective batch size = 16
EPOCHS         = 2    # fits within ~2-hr free tier limit
LR             = 2e-4
WARMUP_STEPS   = 20
VAL_SPLIT      = 0.1  # 10% held out for validation
SEED           = 42

# W&B
WANDB_PROJECT  = "mlops-sg-law"
WANDB_RUN_NAME = "qwen2.5-3b-qlora-sg-criminal-law"

## 2. Login to W&B

In [ ]:
import wandb
import os

# Get your free API key from: https://wandb.ai/authorize
WANDB_API_KEY = "paste-your-key-here"  # ← replace this

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)

## 3. Upload training data

Upload `data/qa_pairs.jsonl` from your local machine using the cell below, or mount Google Drive.

In [ ]:
# Option A: Upload directly
from google.colab import files
uploaded = files.upload()  # select qa_pairs.jsonl from your machine
DATA_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_PATH}")

In [ ]:
# Option B: Mount Google Drive (if you uploaded qa_pairs.jsonl there)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/qa_pairs.jsonl'

## 4. Load and inspect dataset

In [ ]:
import json
from collections import Counter

with open(DATA_PATH) as f:
    raw = [json.loads(line) for line in f if line.strip()]

print(f"Total Q&A pairs: {len(raw)}")

# Domain distribution
areas = Counter(e["metadata"]["area_of_law"] for e in raw)
print("\nDomain distribution:")
for area, count in areas.most_common():
    print(f"  {area:<35} {count}")

# Sample
print("\n--- Sample Q&A pair ---")
sample = raw[0]
for msg in sample["messages"]:
    print(f"[{msg['role'].upper()}]")
    print(msg["content"][:300])
    print()

## 5. Load base model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # auto-detect
    load_in_4bit=True,   # QLoRA — required to fit 7B on T4
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # critical for fitting 7B on T4
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

## 7. Prepare dataset

In [ ]:
from datasets import Dataset
import random

random.seed(SEED)
random.shuffle(raw)

split = int(len(raw) * (1 - VAL_SPLIT))
train_data = raw[:split]
val_data   = raw[split:]
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")


def format_chat(example):
    """Apply Qwen2.5 chat template and tokenize."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_dataset = Dataset.from_list([{"messages": e["messages"]} for e in train_data])
val_dataset   = Dataset.from_list([{"messages": e["messages"]} for e in val_data])

train_dataset = train_dataset.map(format_chat)
val_dataset   = val_dataset.map(format_chat)

print("\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

## 8. Train

In [ ]:
import os
os.environ["WANDB_PROJECT"]  = WANDB_PROJECT
os.environ["WANDB_RUN_NAME"] = WANDB_RUN_NAME
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # reduce OOM from memory fragmentation

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        learning_rate=LR,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="wandb",
        run_name=WANDB_RUN_NAME,
        seed=SEED,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        weight_decay=0.01,
    ),
)

trainer.train()

## 9. Before vs After evaluation

In [ ]:
TEST_QUERIES = [
    "What is the mandatory minimum sentence for drug trafficking under MDA s 5 if the accused is found to have given substantive assistance to CNB?",
    "What sentencing framework applies to rape offences under s 375 Penal Code in Singapore?",
    "What is the test for adducing fresh evidence on a criminal appeal in Singapore?",
    "What elements must the prosecution prove for a charge of culpable homicide not amounting to murder under s 299 Penal Code?",
    "What are the key mitigating factors a Singapore court considers when sentencing a first-time drug offender?",
]

print("=" * 60)
print("FINE-TUNED MODEL OUTPUTS")
print("=" * 60)

model.eval()
results = []
for query in TEST_QUERIES:
    messages = [
        {"role": "system", "content": "You are an expert Singapore criminal law advisor."},
        {"role": "user",   "content": query},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).cuda()

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=400,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    results.append({"query": query, "response": response})

    print(f"\nQ: {query}")
    print(f"A: {response}")
    print("-" * 60)

# Log sample outputs to W&B
wandb.log({
    "eval/sample_outputs": wandb.Table(
        columns=["query", "response"],
        data=[[r["query"], r["response"]] for r in results],
    )
})

## 10. Save LoRA adapters + export to GGUF for Ollama

In [ ]:
# Save LoRA adapters (~100–150 MB for 7B)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters saved to {OUTPUT_DIR}/")

In [ ]:
# Export merged model to GGUF (Q4_K_M quantization) for Ollama
# This merges the LoRA weights into the base model and quantizes
model.save_pretrained_gguf(
    "sg-law-3b-gguf",
    tokenizer,
    quantization_method="q4_k_m",  # ~2 GB output for 3B
)
print("GGUF model saved to sg-law-3b-gguf/")

In [ ]:
# Download the GGUF file to your local machine
import glob
gguf_files = glob.glob("sg-law-3b-gguf/*.gguf")
print(f"GGUF files: {gguf_files}")

from google.colab import files
for f in gguf_files:
    print(f"Downloading {f}...")
    files.download(f)

In [ ]:
# Also download the LoRA adapters
import shutil
shutil.make_archive("lora_adapters_3b", "zip", OUTPUT_DIR)
files.download("lora_adapters_3b.zip")

In [ ]:
wandb.finish()
print("Training complete. Check your W&B dashboard for logs.")

## 11. Load fine-tuned model in Ollama (run locally after downloading GGUF)

After downloading the `.gguf` file, create a `Modelfile` and register it with Ollama:

```bash
# Create Modelfile
cat > Modelfile_3b << 'EOF'
FROM ./sg-law-qwen2.5-3b-q4_k_m.gguf

SYSTEM "You are an expert Singapore criminal law advisor with deep knowledge of the Penal Code, Misuse of Drugs Act, Criminal Procedure Code 2010, and Singapore appellate case law."

PARAMETER temperature 0.3
PARAMETER num_predict 2048
PARAMETER repeat_penalty 1.4
EOF

# Register with Ollama
ollama create sg-law-qwen2.5-3b -f Modelfile_3b

# Test it
ollama run sg-law-qwen2.5-3b "What is the sentencing framework for drug trafficking under MDA s 5?"
```

Then select `sg-law-qwen2.5-3b` from the Streamlit sidebar model dropdown.